# Malasanita in Italia — Preanalysis v1
**Anno 2022 | Fonti: Ministero della Salute (A, C) + ISTAT (D)**

**Domanda:** Le regioni con meno personale sanitario hanno tassi di mortalita piu alti?

Questa preanalysis esplora la relazione tra dotazione di personale (medici di base e personale ospedaliero) e mortalita regionale su **21 regioni/PA** per l'anno 2022.
Il proxy usato per la mortalita e la **mortalita totale 30+** — non mortalita evitabile (v1). v2 usera Euro-2013 proxy (amenable + preventable). Vedi nota metodologica.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

cwd = Path.cwd().resolve()
project_root = next(
    (
        path for path in [cwd, *cwd.parents]
        if (path / "sources").exists() and (path / "compose").exists()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Non trovo la root di malasanita-struttura-mortalita risalendo da cwd")

repo_root = project_root.parents[2]
MART_PATH = repo_root / "out" / "data" / "mart" / "malasanita_a_strutture_asl" / "2022" / "mart_compose_regioni.parquet"

df = pd.read_parquet(MART_PATH)
print(f"Regioni/PA caricate: {len(df)} | Anno: {df['anno'].iloc[0]}")
print(f"Join C ok: {df['join_c_ok'].sum()}/21 | Join D ok: {df['join_d_ok'].sum()}/21")

## Panoramica regionale 2022

Tabella ordinata per mortalitÃ  decrescente.

In [ ]:
cols_display = [
    'regione',
    'medici_mmg_per_100k',
    'infermieri_per_100k',
    'personale_osp_per_100k',
    'posti_letto_previsti_per_100k',
    'decessi_evitabili_30plus_per_100k_pop_totale',
]

tab = (
    df[cols_display]
    .sort_values('decessi_evitabili_30plus_per_100k_pop_totale', ascending=False)
    .reset_index(drop=True)
)
tab.columns = ['Regione', 'MMG/100k', 'Infermieri/100k', 'Pers.osp/100k', 'PL/100k', 'Evitabili30+/100k']
tab.index += 1
tab

## Numeri chiave

In [ ]:
mort = df['decessi_evitabili_30plus_per_100k_pop_totale']
mmg  = df['medici_mmg_per_100k']
inf  = df['infermieri_per_100k']

r_mort_max = df.loc[mort.idxmax(), 'regione']
r_mort_min = df.loc[mort.idxmin(), 'regione']
r_mmg_max  = df.loc[mmg.idxmax(),  'regione']
r_mmg_min  = df.loc[mmg.idxmin(),  'regione']
r_inf_max  = df.loc[inf.idxmax(),  'regione']
r_inf_min  = df.loc[inf.idxmin(),  'regione']

print("â”€â”€ MORTALITÃ€ 30+ per 100k (proxy totale) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€")
print(f"  PiÃ¹ alta: {r_mort_max:<30} {mort.max():.0f}")
print(f"  PiÃ¹ bassa: {r_mort_min:<29} {mort.min():.0f}")
print(f"  Scarto max-min: {mort.max()-mort.min():.0f} decessi per 100k")
print(f"  Media nazionale: {mort.mean():.0f}")
print()
print("â”€â”€ MMG per 100k â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€")
print(f"  PiÃ¹ alto: {r_mmg_max:<30} {mmg.max():.1f}")
print(f"  PiÃ¹ basso: {r_mmg_min:<29} {mmg.min():.1f}")
print()
print("â”€â”€ INFERMIERI per 100k â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€")
print(f"  PiÃ¹ alto: {r_inf_max:<30} {inf.max():.0f}")
print(f"  PiÃ¹ basso: {r_inf_min:<29} {inf.min():.0f}")

## Scatter â€” Medici di base (MMG) vs mortalitÃ  30+

Asse X: MMG per 100k residenti (fonte A) | Asse Y: decessi 30+ per 100k popolazione totale (fonte D)

**Atteso:** correlazione negativa â€” piÃ¹ MMG, meno mortalitÃ .

In [ ]:
def sigla(nome):
    """Abbreviazione regione per etichetta scatter."""
    m = {
        'Valle': 'VDA', 'Bolzano': 'BZ', 'Trento': 'TN',
        'Friuli': 'FVG', 'Emilia': 'ER', 'Toscana': 'TOS',
        'Umbria': 'UMB', 'Liguria': 'LIG', 'Lombardia': 'LOM',
        'Piemonte': 'PIE', 'Veneto': 'VEN', 'Marche': 'MAR',
        'Lazio': 'LAZ', 'Abruzzo': 'ABR', 'Molise': 'MOL',
        'Campania': 'CAM', 'Puglia': 'PUG', 'Basilicata': 'BAS',
        'Calabria': 'CAL', 'Sicilia': 'SIC', 'Sardegna': 'SAR',
    }
    for k, v in m.items():
        if k in nome:
            return v
    return nome[:3].upper()


def scatter_plot(ax, x_col, y_col, x_label, title, df):
    x = df[x_col]
    y = df[y_col]

    ax.scatter(x, y, color='#2166ac', alpha=0.85, s=60, zorder=3)

    # Linea di regressione
    m, b = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, m * x_line + b, color='#d62728', linewidth=1.2,
            linestyle='--', alpha=0.7, label=f'Regressione (m={m:.0f})')

    # Correlazione
    r = np.corrcoef(x, y)[0, 1]
    ax.text(0.97, 0.97, f'r = {r:.2f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=9, color='#555555')

    # Etichette regioni
    for _, row in df.iterrows():
        ax.annotate(
            sigla(row['regione']),
            (row[x_col], row[y_col]),
            textcoords='offset points', xytext=(5, 3),
            fontsize=7.5, color='#333333'
        )

    ax.set_xlabel(x_label, fontsize=9)
    ax.set_ylabel('Decessi 30+ per 100k (proxy mortalitÃ )', fontsize=9)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, linestyle=':')


fig, ax = plt.subplots(figsize=(9, 6))
scatter_plot(
    ax,
    x_col='medici_mmg_per_100k',
    y_col='decessi_evitabili_30plus_per_100k_pop_totale',
    x_label='Medici di base (MMG) per 100k residenti',
    title='MMG per 100k vs MortalitÃ  totale 30+ per 100k â€” Regioni italiane 2022',
    df=df
)
plt.tight_layout()
plt.show()

## Scatter â€” Infermieri (personale ospedaliero) vs mortalitÃ  30+

Asse X: infermieri per 100k (fonte C) | Asse Y: decessi 30+ per 100k (fonte D)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
scatter_plot(
    ax,
    x_col='infermieri_per_100k',
    y_col='decessi_evitabili_30plus_per_100k_pop_totale',
    x_label='Infermieri per 100k residenti',
    title='Infermieri per 100k vs MortalitÃ  totale 30+ per 100k â€” Regioni italiane 2022',
    df=df
)
plt.tight_layout()
plt.show()

## Scatter â€” Posti letto per 100k vs mortalitÃ  30+

Asse X: posti letto previsti per 100k (fonte C) | Asse Y: decessi 30+ per 100k (fonte D)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
scatter_plot(
    ax,
    x_col='posti_letto_previsti_per_100k',
    y_col='decessi_evitabili_30plus_per_100k_pop_totale',
    x_label='Posti letto previsti per 100k residenti',
    title='Posti letto per 100k vs MortalitÃ  totale 30+ per 100k â€” Regioni italiane 2022',
    df=df
)
plt.tight_layout()
plt.show()

## Nota di lettura

Questa e una lettura descrittiva intermedia. La **risposta finale alla domanda analitica** e riportata solo nell'ultima sezione del notebook, dopo tutte le visual aggiuntive.


---


## Visual aggiuntive per rispondere meglio alla domanda

Le visual sotto puntano a una risposta piu diretta: quanto la mortalita 30+ si muove insieme alla dotazione sanitaria regionale?


### 1) Scatter chiave: personale ospedaliero totale vs mortalita


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
scatter_plot(
    ax,
    x_col='personale_osp_per_100k',
    y_col='decessi_evitabili_30plus_per_100k_pop_totale',
    x_label='Personale ospedaliero totale per 100k residenti',
    title='Personale ospedaliero per 100k vs Mortalita evitabile 30+ per 100k - Regioni italiane 2022 (Euro-2013 proxy)',
    df=df
)
plt.tight_layout()
plt.show()


### 2) Quadranti: alta/bassa dotazione vs alta/bassa mortalita


In [ ]:
x_col = 'personale_osp_per_100k'
y_col = 'decessi_evitabili_30plus_per_100k_pop_totale'
x = df[x_col]
y = df[y_col]
x_med = x.median()
y_med = y.median()

quad = df.copy()
quad['quadrante'] = np.select(
    [
        (quad[x_col] >= x_med) & (quad[y_col] < y_med),
        (quad[x_col] >= x_med) & (quad[y_col] >= y_med),
        (quad[x_col] < x_med) & (quad[y_col] >= y_med),
        (quad[x_col] < x_med) & (quad[y_col] < y_med),
    ],
    ['Alta dotazione / Bassa mortalita', 'Alta dotazione / Alta mortalita', 'Bassa dotazione / Alta mortalita', 'Bassa dotazione / Bassa mortalita'],
    default='Non classificato'
)

colors = {
    'Alta dotazione / Bassa mortalita': '#1a9850',
    'Alta dotazione / Alta mortalita': '#66bd63',
    'Bassa dotazione / Alta mortalita': '#d73027',
    'Bassa dotazione / Bassa mortalita': '#fdae61',
}

fig, ax = plt.subplots(figsize=(10, 7))
for q_name, g in quad.groupby('quadrante'):
    ax.scatter(g[x_col], g[y_col], s=70, alpha=0.9, color=colors[q_name], label=f"{q_name} (n={len(g)})", zorder=3)
    for _, r in g.iterrows():
        ax.annotate(sigla(r['regione']), (r[x_col], r[y_col]), textcoords='offset points', xytext=(4, 3), fontsize=7.5)

ax.axvline(x_med, color='#555555', linestyle='--', linewidth=1)
ax.axhline(y_med, color='#555555', linestyle='--', linewidth=1)
ax.text(0.01, 0.98, f'Mediana personale osp: {x_med:.0f}', transform=ax.transAxes, va='top', fontsize=8, color='#555555')
ax.text(0.01, 0.93, f'Mediana evitabile 30+: {y_med:.0f}', transform=ax.transAxes, va='top', fontsize=8, color='#555555')
ax.set_xlabel('Personale ospedaliero totale per 100k')
ax.set_ylabel('Decessi evitabili 30+ per 100k (Euro-2013 proxy)') 
ax.set_title('Mappa a quadranti - Dotazione sanitaria vs mortalita evitabile (2022, Euro-2013 proxy)', fontweight='bold')
ax.grid(True, alpha=0.25, linestyle=':')
ax.legend(fontsize=8, loc='best')
plt.tight_layout()
plt.show()

quad_counts = quad['quadrante'].value_counts().rename_axis('Quadrante').to_frame('Regioni')
quad_counts


### 3) Rank-dumbbell: ranking dotazione vs ranking mortalita


In [ ]:
rank_df = df[['regione', 'personale_osp_per_100k', 'decessi_evitabili_30plus_per_100k_pop_totale']].copy()
rank_df['rank_personale'] = rank_df['personale_osp_per_100k'].rank(method='min', ascending=False)
rank_df['rank_mortalita'] = rank_df['decessi_evitabili_30plus_per_100k_pop_totale'].rank(method='min', ascending=False)
rank_df = rank_df.sort_values('rank_mortalita')

fig, ax = plt.subplots(figsize=(10, 9))
y_pos = np.arange(len(rank_df))
for i, (_, r) in enumerate(rank_df.iterrows()):
    ax.plot([r['rank_personale'], r['rank_mortalita']], [i, i], color='#999999', linewidth=1.2, zorder=1)
    ax.scatter(r['rank_personale'], i, color='#2166ac', s=45, zorder=3)
    ax.scatter(r['rank_mortalita'], i, color='#d73027', s=45, zorder=3)

ax.set_yticks(y_pos)
ax.set_yticklabels(rank_df['regione'])
ax.invert_yaxis()
ax.set_xlabel('Ranking (1 = valore piu alto)')
ax.set_title('Dumbbell ranking - Personale ospedaliero (blu) vs Mortalita evitabile 30+ (rosso, Euro-2013)', fontweight='bold')
ax.grid(axis='x', alpha=0.2, linestyle=':')
legend_handles = [
    mpatches.Patch(color='#2166ac', label='Rank personale ospedaliero'),
    mpatches.Patch(color='#d73027', label='Rank evitabile 30+ (Euro-2013)'),
]
ax.legend(handles=legend_handles, loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

rank_df[['regione', 'rank_personale', 'rank_mortalita']].head(10)


### 4) Heatmap correlazioni tra indicatori principali


In [ ]:
corr_cols = [
    'medici_mmg_per_100k',
    'infermieri_per_100k',
    'personale_osp_per_100k',
    'posti_letto_previsti_per_100k',
    'decessi_evitabili_30plus_per_100k_pop_totale',
]
corr = df[corr_cols].corr()

labels = [
    'MMG/100k',
    'Infermieri/100k',
    'Personale osp/100k',
    'Posti letto/100k',
    'Evitabile 30+/100k',
]

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=35, ha='right')
ax.set_yticklabels(labels)
ax.set_title('Heatmap correlazioni Pearson - indicatori chiave 2022 (Euro-2013 proxy)', fontweight='bold')

for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        ax.text(j, i, f'{corr.values[i, j]:.2f}', ha='center', va='center', fontsize=8, color='black')

cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Correlazione r')
plt.tight_layout()
plt.show()
corr


## Risposta sintetica alla domanda analitica

Con il dataset 2022 e il proxy di **mortalita totale 30+**, le correlazioni di Pearson tra dotazione sanitaria e mortalita sono **tutte positive**:

| Indicatore | r con mortalita 30+ |
|---|---|
| MMG/100k | **+0.52** |
| Infermieri/100k | **+0.39** |
| Personale ospedaliero/100k | **+0.04** |
| Posti letto/100k | **+0.18** |

Il segno opposto a quello atteso: piu personale sanitario corrisponde a piu mortalita, non meno.

**La risposta con questo proxy e: no, o almeno non nel senso ipotizzato.**

### Perche? Il confondente demografico

Le regioni con mortalita 30+ piu alta (Liguria, Piemonte, FVG) sono anche le **piu anziane d'Italia**.
Hanno piu infrastruttura sanitaria *perche* ne hanno bisogno, e piu mortalita *perche* la popolazione e vecchia.
La mortalita totale 30+ non e age-adjusted: cattura struttura demografica, non performance del sistema sanitario.

### Outlier significativi

- **Liguria**: mortalita #1 (peggiore), personale osp rank #20 — unico caso coerente con l'ipotesi
- **Molise**: mortalita #2, personale osp rank #1 — contraddizione esplicita
- **Bolzano/Trento**: mortalita piu bassa, dotazione variabile

### v2 — decisione metodologica presa

Framework adottato: **Euro-2013 proxy** (Eurostat, Avoidable mortality — amenable and preventable).

12 cause mappate dalla fonte D: Sepsi, Tumori colon/retto, Tumori polmone, Tumore seno, Diabete,
Malattie ischemiche cuore, Malattie cerebrovascolari, Malattie ipertensive, Influenza e Polmonite,
Malattie croniche vie respiratorie, Cirrosi/epatite cronica, Cause esterne.

Mart D aggiornato con il nuovo filtro. Prossimo passo: rieseguire la pipeline e confrontare i risultati.

---

*Nota metodologica:*
*- Questa preanalysis (v1) usa un proxy di mortalita totale (30+, cod_causa=25, tutte le cause) non standardizzato per eta.*
*- cod_classe_eta=9 = fascia 30 anni e oltre (perimetro dichiarato esplicitamente).*
*- I risultati v1 non sono interpretabili come valutazione causale della performance sanitaria regionale.*
*- v2 usera mortalita evitabile Euro-2013 proxy: tasso grezzo 30+ su 12 cause amenable/preventable.*
*  Il tasso risultante e grezzo (non age-standardized) — la distinzione e mantenuta nel naming del campo.*
*- I dati Ministero della Salute sono fermi al 2022 (gap temporale strutturale): parte della narrativa, documentato.*
*- Emilia-Romagna esclusa dall'analisi principale (benchmark metodologico opzionale, accordo Discussion #99).*